In [2]:
import molli as ml
from molli.visual._pyvista import draw_wireframe, plot_descriptor
import numpy as np
from tqdm import tqdm

lib = ml.ConformerLibrary("bpa_aligned.clib")

with lib.reading():
    keys = list(lib.keys())

In [4]:
import pyvista as pv
from pyvista.plotting.plotter import Plotter
from PIL import ImageColor

plt = pv.Plotter()

with lib.reading():
    for k in tqdm(keys[:50]):
        color_darkness = 50
        opacity = 0.005
        s = lib[k]
        
        lines = []
        for i, conf in enumerate(s):
            for b in s.bonds:
                i1, i2 = map(s.get_atom_index, (b.a1, b.a2))
                lines.extend((2, i1 + s.n_atoms * i, i2 + s.n_atoms * i))

        colors = [ImageColor.getrgb(a.color_cpk) for a in s.atoms] * s.n_conformers
        colors = np.clip((np.array(colors) - color_darkness) / 255, 0, 1)
        
        data = pv.PolyData(
            s.coords.reshape((s.n_atoms * s.n_conformers, 3)),
            lines=lines,
        )

        plt.add_mesh(
            data,
            rgb=True,
            scalars=colors,
            interpolate_before_map=False,
            line_width=2,
            opacity=opacity,
        )
    
        plt.add_points(
            data,
            rgb=True,
            scalars=colors,
            interpolate_before_map=False,
            point_size=3,
            opacity=opacity,
        )
    
plt.show()
# plt.export_gltf('test.gltf')
plt.export_vtksz('test.vtksz')

100%|███████████████████████████████████████████| 50/50 [00:02<00:00, 20.35it/s]


Widget(value='<iframe src="http://localhost:54002/index.html?ui=P_0x14d314230_2&reconnect=auto" class="pyvista…

'test.vtksz'

In [8]:
from sklearn.decomposition import PCA
import h5py

with h5py.File("bpa_aligned_grid10.hdf5") as f:
    grid = np.asarray(f["grid"])

DESC_NAME = "aso"
desc_all = np.empty((len(keys), grid.shape[0]))
with h5py.File(f"bpa_aligned_grid10_{DESC_NAME}.hdf5") as f:
    for i, k in enumerate(keys):
        desc_all[i] = f[k]

pca = PCA(n_components=30).fit(desc_all)
desc_pca1 = pca.components_[0]

data = pv.PolyData(grid)
desc_norm = desc_pca1 / np.max(np.abs(desc_pca1))

In [9]:
plt1 = pv.Plotter()

name=f"PCA({DESC_NAME.upper()}, 1)"

pd = pv.PolyData(grid)
pd.point_data[name] = desc_norm
gly = pd.glyph(geom=pv.Sphere(0.5), scale=name)
plt1.add_mesh(gly, scalars=name, opacity=0.75, scalar_bar_args={
  "position_x": 0.25,
  "width": 0.5,
  "label_font_size": 16,
})

plt1.enable_anti_aliasing("fxaa")

plt1.camera_position = [
    (29.96358042372618, 4.0391504855202776, 0.08371776837089945),
    (-0.007199933881796489, 0.27696410129923404, -0.2762142895658706),
    (0.04262366414585409, -0.42601280695171206, 0.9037125159960466),
]

plt1.screenshot(
    filename=f"PCA_{DESC_NAME.upper()}_1.png",
    transparent_background=None,
    window_size=(1600, 1200),
    scale=2,
)

plt1.show()


/opt/homebrew/Caskroom/miniconda/base/envs/test/lib/python3.12/site-packages/pyvista/core/filters/data_set.py:2320: UserWarning: No vector-like data to use for orient. orient will be set to False.
  warnings.warn("No vector-like data to use for orient. orient will be set to False.")


Widget(value='<iframe src="http://localhost:59620/index.html?ui=P_0x2adaee4e0_3&reconnect=auto" class="pyvista…